# 02 - Model Training

Goal: generate the synthetic-label training set, train the baseline RandomForestClassifier, and evaluate it.

**Important:** the training label here is a documented, rule-based heuristic (see `scripts/generate_training_data.py:heuristic_label`), not a real clinical judgment -- no dyslexia-labeled dataset is used. This demonstrates the ML pipeline mechanics, not a validated screening classifier. See `data/generated/README.md`.

In [1]:
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path('..').resolve()

def run(*args):
    result = subprocess.run(
        [sys.executable, *args], cwd=REPO_ROOT,
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'{args} failed with code {result.returncode}')

run('-m', 'scripts.generate_training_data')

Wrote 200 rows to /private/tmp/claude-501/-Users-krisharathod-Desktop/56941ab0-c436-452e-accc-b3b3744394d8/scratchpad/neuroaid-ai/data/generated/training_data.csv



In [2]:
import pandas as pd
df = pd.read_csv(REPO_ROOT / 'data' / 'generated' / 'training_data.csv')
print(df.shape)
df[['wpm', 'word_error_rate', 'phoneme_mismatch_proxy', 'pause_ratio', 'label']].head()

(200, 33)


,wpm,word_error_rate,phoneme_mismatch_proxy,pause_ratio,label
0,147.039775,0.266667,0.333333,0.182287,0
1,77.826646,0.466667,0.227273,0.176206,1
2,222.970343,0.466667,0.287879,0.191049,0
3,109.192966,0.533333,0.272727,0.185012,0
4,109.071632,0.466667,0.545455,0.178346,1


In [3]:
run('-m', 'scripts.train_model')

              precision    recall  f1-score   support

           0       0.82      0.96      0.89        28
           1       0.86      0.50      0.63        12

    accuracy                           0.82        40
   macro avg       0.84      0.73      0.76        40
weighted avg       0.83      0.82      0.81        40

Saved model to /private/tmp/claude-501/-Users-krisharathod-Desktop/56941ab0-c436-452e-accc-b3b3744394d8/scratchpad/neuroaid-ai/models/risk_model.joblib



The trained model is saved to `models/risk_model.joblib` and is what `src/model/predict.py` (and the Flask API's `/api/v1/screen` endpoint) loads to produce a `risk_assessment` in responses.